## 利用逻辑回归实现乳腺癌检测

### 1. Loading Libraries

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### 2. Loading dataset

In [ ]:
from sklearn.datasets import load_breast_cancer

data_bunch = load_breast_cancer()
data = pd.DataFrame(data_bunch.data, columns=data_bunch.feature_names)  # type: ignore
data['target'] = data_bunch.target # type: ignore

print(data.head())
data.info()

### 3. Processing Dataset

In [ ]:
X = data.drop('target', axis=1)
y = data['target']
print(X.shape, y.shape)

print("===== 类别分布检查 =====")
class_0 = (y == 0).sum()
class_1 = (y == 1).sum()
print(f"类别 0 (恶性): {class_0} 样本 ({class_0/len(y)*100:.1f}%)")
print(f"类别 1 (良性): {class_1} 样本 ({class_1/len(y)*100:.1f}%)")
if class_0 / len(y) < 0.4:
    print("数据集存在类别不平衡，但比例可接受")
else:
    print("数据集类别分布较为均衡")

### 4. Splitting data for training and testing

In [ ]:
X_train, X_, y_train, y_ = train_test_split(X, y, test_size=0.4, random_state=42)
X_cv, X_test, y_cv, y_test = train_test_split(X_, y_, test_size=0.5, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_cv = scaler.transform(X_cv)
X_test = scaler.transform(X_test)

print(X_train.shape, y_train.shape)
print(X_cv.shape, y_cv.shape)
print(X_test.shape, y_test.shape)

### 5. Training logisticRegression and Printing losses

In [ ]:
# 1. 训练逻辑回归模型
model = LogisticRegression(
    max_iter=5000,    # 足够大，确保收敛
    solver='saga',
    random_state=42
)
model.fit(X_train, y_train)  # 只用训练集训练


# 2. 在 训练集 / 验证集(dev) / 测试集 上评估
y_pred_train = model.predict(X_train)
train_acc = accuracy_score(y_train, y_pred_train)

y_pred_cv = model.predict(X_cv)
cv_acc = accuracy_score(y_cv, y_pred_cv)

# 输出结果
print("===== 模型评估结果 =====")
print(f"训练集准确率: {train_acc:.4f}")
print(f"验证集准确率: {cv_acc:.4f}")

### 6. Making Predictions

In [ ]:
y_pred_test = model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred_test)

print("===== 测试集评估结果 =====")
print("混淆矩阵:")
print(confusion_matrix(y_test, y_pred_test))
print("\n分类报告:")
print(classification_report(y_test, y_pred_test))
print(f"\n测试集准确率: {test_acc:.4f}")